# Classes and Objects

*Run a cell with ▶ (or Shift+Enter): Sphinx compiles and runs it on your computer. Change the code and run it again: this notebook is your own copy.*

**After this unit you can:**
- Write a class with private fields, a constructor and methods, and create objects from it.
- Use a record for plain data and an enum for a fixed set of values.
- Share behaviour with inheritance and interfaces, and override methods such as toString.

**In this notebook**, one idea at a time (run each cell before moving on):
1. A class describes a kind of object
2. Constructors: setting an object up
3. Variables hold references
4. private: protecting the fields
5. toString: how an object prints
6. Records: classes for plain data
7. Enums: a fixed set of values
8. Interfaces and inheritance

**Exercises** in this notebook end with `check(...)` lines: minimal tests that print ✓ when your code is right. Run the next cell once first, so they can use `check`.

In [ ]:
// shared: check(...) prints ✓ or ✗; the exercises below use it
void check(String what, Object actual, Object expected) {
    if (Objects.equals(actual, expected)) {
        IO.println("✓ " + what);
    } else {
        IO.println("✗ " + what + ": expected " + expected + ", got " + actual);
    }
}

## 1. A class describes a kind of object

So far your values were numbers and text. A **class** lets you create your own kind of value. It describes what each object **knows** (its **fields**, variables inside it) and what it **does** (its **methods**). `new` creates an object from the class, and each object has its own copy of the fields. Use a dot to reach them: `a.count`, `a.add()`.

In [ ]:
class Counter {
    int count;          // a field: each Counter has its own

    void add() {        // a method: works on this object's field
        count++;
    }
}

void main() {
    Counter a = new Counter();
    Counter b = new Counter();
    a.add();
    a.add();
    b.add();
    IO.println("a: " + a.count + ", b: " + b.count);
}

🧪 **Try it:** add a method `void reset()` that puts `count` back to 0, and call it on `a` only.

## 2. Constructors: setting an object up

A **constructor** runs when `new` creates the object, to give the fields their first values. It has the class's name and no return type. Inside, `this.name` is the field, while `name` alone is the parameter: `this.name = name;` copies one into the other.

In [ ]:
class Pet {
    String name;
    int age;

    Pet(String name, int age) {
        this.name = name;
        this.age = age;
    }

    String describe() {
        return name + " (" + age + (age == 1 ? " year)" : " years)");
    }
}

void main() {
    Pet rex = new Pet("Rex", 3);
    Pet mia = new Pet("Mia", 1);
    IO.println(rex.describe());
    IO.println(mia.describe());
}

🧪 **Try it:** remove `this.` from `this.name = name;` and run: the field stays `null`, because you copied the parameter into itself. Then try `new Pet("Bob")` with one argument.

## 3. Variables hold references

An object lives in the heap, and the variable holds an arrow to it, just as with arrays. Two `new`s make two objects, even with equal values. `Seat s3 = s1;` creates no object: both variables point to the same one. That's why `==` asks "the same object?", while `equals` asks "the same values?". The slides show it step by step.

#### 🎞️ Objects, references and equality

![Two news make two objects even with equal values; assigning shares one object; == compares objects and equals compares values](objects-memory-steps.svg)

In [ ]:
record Seat(char row, int number) {}

void main() {
    Seat s1 = new Seat('A', 1);
    Seat s2 = new Seat('A', 1);
    Seat s3 = s1;
    IO.println(s1 == s2);
    IO.println(s1.equals(s2));
    IO.println(s1 == s3);
}

🧪 **Try it:** predict the three lines before running. (`record` comes in step 6: it writes `equals` for you.)

### ✏️ Exercise: Change an object through a method

A method gets a copy of the reference, not of the object. Make `fill` put v in the box it gets: the caller's Box changes too.

In [ ]:
class Box {
    int value = -1;
}

/** Fills a new box with v, then reads it back from the caller's side. */
int filledWith(int v) {
    Box mine = new Box();
    fill(mine, v);
    return mine.value;
}

void fill(Box box, int v) {
    // TODO: set box.value
}

void main() {
    check("the caller sees 7", filledWith(7), 7);
    check("and then 0", filledWith(0), 0);
}

<details>
<summary>Show an answer</summary>

```java
class Box {
    int value = -1;
}

/** Fills a new box with v, then reads it back from the caller's side. */
int filledWith(int v) {
    Box mine = new Box();
    fill(mine, v);
    return mine.value;
}

void fill(Box box, int v) {
    box.value = v;
}

void main() {
    check("the caller sees 7", filledWith(7), 7);
    check("and then 0", filledWith(0), 0);
}
```

</details>

## 4. private: protecting the fields

If any code can change a field, nothing stops an invalid value, such as a thermostat at 500 degrees. Mark the fields `private`: only the class's own methods can touch them. Other code goes through methods that check the values. This is called **encapsulation**.

In [ ]:
class Thermostat {
    private int target = 20;

    void setTarget(int degrees) {
        if (degrees < 10 || degrees > 30) {
            IO.println("Ignored: " + degrees + " is out of range");
            return;
        }
        target = degrees;
    }

    int getTarget() {
        return target;
    }
}

void main() {
    Thermostat t = new Thermostat();
    t.setTarget(23);
    t.setTarget(500);
    IO.println("Target: " + t.getTarget());
}

🧪 **Try it:** add a method `void warmer()` that raises the target by 1 degree, but never above 30.

## 5. toString: how an object prints

Printing an object calls its `toString()` method. Without one you get the class name and a code, like `Pet@1b6d3586`. Write your own to choose the text. `@Override` tells the compiler you're replacing a method every class inherits, and it complains if the name is misspelled.

In [ ]:
class Pet {
    private final String name;
    private final int age;

    Pet(String name, int age) {
        this.name = name;
        this.age = age;
    }

    @Override
    public String toString() {
        return name + ", " + age;
    }
}

void main() {
    Pet rex = new Pet("Rex", 3);
    IO.println(rex);
    IO.println("My pet: " + rex);
}

🧪 **Try it:** delete the `toString` method and run: that's the default text. Then misspell it as `tostring` with `@Override` still on top and read the error.

## 6. Records: classes for plain data

Many classes just hold data. A **record** writes all the usual code for you from one line: `record Song(String title, int seconds) {}` gives a constructor, the accessors `title()` and `seconds()`, `toString`, and an `equals` that compares the values. Its fields can't change after creation. You can still add methods inside the braces.

In [ ]:
record Song(String title, int seconds) {
    String length() {
        return seconds / 60 + ":" + String.format("%02d", seconds % 60);
    }
}

void main() {
    Song intro = new Song("Intro", 95);
    IO.println(intro);
    IO.println(intro.title() + " lasts " + intro.length());
    IO.println(intro.equals(new Song("Intro", 95)));
}

🧪 **Try it:** add a field `String artist` to the record and fix the `new Song(...)` calls. Then try `intro.seconds = 10;`: records can't change.

### ✏️ Exercise: An object's label

A Seat has a row letter and a number. Return its label, such as `A1`, from the record's accessors.

In [ ]:
record Seat(char row, int number) {}

String label(Seat seat) {
    // TODO: join seat.row() and seat.number()
    return "";
}

void main() {
    check("A1", label(new Seat('A', 1)), "A1");
    check("C12", label(new Seat('C', 12)), "C12");
}

<details>
<summary>Show an answer</summary>

```java
record Seat(char row, int number) {}

String label(Seat seat) {
    return "" + seat.row() + seat.number();
}

void main() {
    check("A1", label(new Seat('A', 1)), "A1");
    check("C12", label(new Seat('C', 12)), "C12");
}
```

</details>

## 7. Enums: a fixed set of values

When a value can only be one of a few options (the days of the week, the suits of a deck), an **enum** lists them. The compiler then refuses anything else, and a `switch` can go through every option. `values()` gives all of them in order.

In [ ]:
enum Direction { NORTH, EAST, SOUTH, WEST }

Direction turnRight(Direction d) {
    return switch (d) {
        case NORTH -> Direction.EAST;
        case EAST -> Direction.SOUTH;
        case SOUTH -> Direction.WEST;
        case WEST -> Direction.NORTH;
    };
}

void main() {
    Direction facing = Direction.NORTH;
    facing = turnRight(facing);
    IO.println("Now facing " + facing);
    for (Direction d : Direction.values()) {
        IO.print(d + " ");
    }
    IO.println();
}

🧪 **Try it:** turn right three more times. Then delete one `case` and read the error: the switch no longer covers every direction (and it needs no `default`, because the enum has only these four).

## 8. Interfaces and inheritance

An **interface** lists methods that several classes promise to have, so code can use any of them the same way. Each class that `implements` it writes its own version. A class can also `extends` another to inherit its fields and methods and change some with `@Override`.

In [ ]:
interface Discount {
    double apply(double price);
}

class Percent implements Discount {
    private final int percent;

    Percent(int percent) {
        this.percent = percent;
    }

    public double apply(double price) {
        return price * (100 - percent) / 100;
    }
}

class Fixed implements Discount {
    public double apply(double price) {
        return Math.max(0, price - 10);
    }
}

void main() {
    List<Discount> offers = List.of(new Percent(20), new Fixed());
    for (Discount d : offers) {
        IO.println(d.getClass().getSimpleName() + ": " + d.apply(30));
    }
}

🧪 **Try it:** write a third discount, `HalfPrice`, and add it to the list: the loop uses it without any other change.

## In short

A **class** describes a kind of object: its **fields** (what it knows) and its **methods** (what it does). `new` creates an object, and the **constructor** sets it up. Keep fields `private`, so other code goes through the methods.

```java
record Song(String title, int seconds) {}

class Playlist {
    private final List<Song> songs = new ArrayList<>();

void add(Song song) {
        songs.add(song);
    }

int totalSeconds() {
        int total = 0;
        for (Song s : songs) {
            total += s.seconds();
        }
        return total;
    }
}

void main() {
    Playlist mix = new Playlist();
    mix.add(new Song("Intro", 95));
    mix.add(new Song("Theme", 210));
    IO.println(mix.totalSeconds() + " seconds");
    IO.println(new Song("Intro", 95));
}
```

A **record** writes the constructor, getters, `toString` and `equals` for you. An `enum` lists fixed values. A class can `extend` another or `implement` an interface, and `@Override` replaces an inherited method.

**Watch out:** two objects with the same values aren't `==`. Compare them with `equals`: records have it already; in a class you write `equals` (and `hashCode`).

## Readings

- [Classes and Objects in Java](https://www.freecodecamp.org/news/the-java-handbook/#heading-classes-and-objects-in-java) (freeCodeCamp, The Java Handbook, 15 min): Classes and objects, methods, constructors, getters and setters, then inheritance and overriding a method, for Shapes, Animals and Payroll. The handbook writes System.out.println inside public static void main; in Sphinx, IO.println and void main() do the same.

## Practice

The notebook covered the basics. Now apply them in the unit's challenges, in the Sphinx sidebar, where Submit runs every test, hidden ones included:

- **Your First Class** · Easy
- **Rectangle Class** · Easy
- **Book with toString()** · Easy
- **Points as Records** · Easy
- **ID Generator (static)** · Easy
- **Bank Account (Encapsulation)** · Medium
- **Coins (enum)** · Medium
- **Shapes (Inheritance)** · Medium
- **Animals (Interfaces)** · Medium
- **Equal Points (equals and hashCode)** · Hard
- **Payroll (Polymorphism)** · Hard
- Test: **Library System** · Hard